<a id="unit-u.0330341eb0c8433c95603983cccc65c7"></a>

<div>
<div>КЕҢЕЙТУ · APACHE DORIS КӨМЕГІМЕН ДЕРЕКТЕР ҚОЙМАСЫН ҚҰРУ</div>
<h1>4–5-модульдердің кеңейтілген зертханалық жұмысы: файлдарды тікелей сұрау, пакеттік жүктеу және Group Commit растаулары</h1>
<p>Мақсатты нұсқа — Doris 4.1.3 · Тәуелсіз ext_* зертханалық кестелері</p>
</div>

[4-модуль](../module04-external-access/course.md) · [5-модуль](../module05-ingestion/course.md) · [Курстың README файлы](../../README.md)

Алдымен 1, 4 және 5-зертханалық жұмыстарды сол курс сынақ ортасында және 4-зертханалық жұмыстың Silo объектілік сақтауын қолданып аяқтаңыз. 4-зертханалық жұмыс көл кестесінің файлын S3 кестелік функциясымен оқып, INSERT INTO SELECT арқылы жүктейді, ал 5-зертханалық жұмыс off_mode режимін Group Commit async_mode режимімен салыстырады; бұл кеңейту Broker Load-ты, әдепкі мәндер мен есептелетін бағандарды салыстыруды, Group Commit үш режимінің растауы мен көрінуін қосады. Ол курстың деректер көлі ортасын тексеріп, дайындайды және сыртқы қызметке қосылмайды. Ол Parquet файлдарын курс тәуелділіктері орнатпайтын PyArrow арқылы жазады: алдымен курс каталогында `.venv/bin/python -m pip install -e '.[extensions]'` орындап, ядроны қайта іске қосыңыз.
Ол тек `ext_file_insert`, `ext_file_broker`, `ext_defaults` және `ext_gc_off_mode/sync_mode/async_mode` кестелерін қайта жасайды, ал объектілік сақтауда тек ағымдағы зертханалық дерекқордың кездейсоқ жолының астына файлдар қосады; басқа файлдарды жоймайды. Зертханалық жұмыстардан бөлек 30–45 минут бөліңіз.
Бір уақытта бір ядрода орындаңыз. Жүктеу сәтсіз болса, алдымен жауабын және `SHOW LOAD` оқыңыз; асинхронды жүктеуді бірден қайта жібермеңіз. Баптау ұяшығы сынақ ортасының BE-сін көрсетеді: `Alive` true және `Version` `doris-4.1.3-rc02-7126cf65d96`.

WWI он тапсырыс файлын орнында сұрау мен екі жүктелген көшірме әр жол бойынша сәйкес келсе, әдепкі және есептелетін бағандарда күтілетін мәндер болса, Group Commit үш режимінің бәрі жалпы сомасы 1400.00 болатын 10 жолмен аяқталса, жұмыс бітеді. Бұл кеңейту Kafka, CDC, файлдарды үздіксіз табуды немесе жоғалған алдын ала жазу журналы дискісінен қалпына келтіруді қамтымайды.


In [ ]:
from pathlib import Path
import sys
COURSE_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))
from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import expect, fixture, normalized
from dw_course.ui import show_sql, show_records, show_response
lab = connect_sandbox()
lab.sql("SHOW BACKENDS", title="The course sandbox's BE", columns=["Host", "Alive", "Version"]);

<a id="unit-u.43027df248ab43199065cfb5b17c29f6"></a>

<a id="1-write-the-ten-order-sample-to-a-separate-parquet-file"></a>

## 1. Он тапсырыстық үлгіні бөлек Parquet файлына жазу

Бұл Iceberg кестесін емес, дербес файл жасайды. Файл 4-зертханалық жұмыстың көл кестесіндегідей он тапсырыстық үлгіні қолданады.
Python хосттың 51900 портына (немесе `DW_LAKE_S3_PORT` орнатқан портқа) қатынайды; Doris курс желісі арқылы course-lake-silo:9000 мекенжайына қатынайды. Оларды шатастырмаңыз.


In [ ]:
from dw_course.environment import port
from dw_course.lakehouse import prepare_lakehouse, ACCESS_KEY, SECRET_KEY
from dw_course.wwi import history_rows, history_ddl, HISTORY_COLUMNS
from decimal import Decimal
from datetime import date
from uuid import uuid4
import boto3
import pyarrow as pa
import pyarrow.parquet as pq
lake_table = prepare_lakehouse(lab, start=True)
records = [dict(zip(HISTORY_COLUMNS, row)) for row in history_rows()]
for row in records:
    row["order_amount"] = Decimal(row["order_amount"])
    row["order_date"] = date.fromisoformat(row["order_date"])
buffer = pa.BufferOutputStream()
pq.write_table(pa.Table.from_pylist(records), buffer)
key = f"{lab.database}/file_demo/{uuid4().hex}/orders.parquet"
s3 = boto3.client("s3",endpoint_url=f"http://127.0.0.1:{port('DW_LAKE_S3_PORT')}",aws_access_key_id=ACCESS_KEY,aws_secret_access_key=SECRET_KEY,region_name="us-east-1")
s3.put_object(Bucket="course-warehouse",Key=key,Body=buffer.getvalue().to_pybytes())
uri = "s3://course-warehouse/" + key
tvf = f'''S3("uri"="{uri}","format"="parquet",
"s3.endpoint"="http://course-lake-silo:9000","s3.region"="us-east-1",
"s3.access_key"="{ACCESS_KEY}","s3.secret_key"="{SECRET_KEY}","use_path_style"="true")'''
projection = "order_id,customer_id,CAST(order_date AS STRING),order_amount,line_count,data_source"
expect(lab.query(f"SELECT {projection} FROM {tvf} ORDER BY order_id"), history_rows())
expect(lab.query(f"SELECT {projection} FROM {lake_table} ORDER BY order_id"), history_rows())
lab.sql(f"SELECT COUNT(*) AS orders,SUM(order_amount) AS amount FROM {tvf}");

<a id="unit-u.e316b99aa1ea45fba969dab9ad2c3eb3"></a>

<a id="2-synchronous-insert-select-and-asynchronous-broker-load"></a>

## 2. Синхронды INSERT SELECT және асинхронды Broker Load

Екі әдіс бөлек кестелерге жазады. `WITH S3` S3 хаттамасын қолданады, қосымша Broker процесі қажет емес.
Broker Load сәтті жіберілгеннен кейін `State=FINISHED` күтіңіз; бас тарту не күту уақытының аяқталуы сәтсіздікті білдіреді, сондықтан сәтті деп хабарлауды жалғастырмаңыз.
Күту уақыты аяқталса, бүкіл ортаны қалпына келтірмей, осы бірегей label-ден бас тарту сұралады. Қате мәліметтері SHOW LOAD ішінде қалады.
Барыс панелі осы бірегей label-ді, ағымдағы `State`/`Progress`, өткен уақытты және 150 секундтық күтудің қалған бөлігін көрсетіп тұрады. Жүктеу тапсырмасының өз күту шегі — 120 секунд. Панель тек FINISHED болғаннан және екі көшірме де әр өріс бойынша салыстырудан өткеннен кейін сәтті аяқталады; сәтсіздік кезінде соңғы бақылауды сақтап, басқа жүктеуді үнсіз жібермейді.


In [ ]:
import time
for table in ("ext_file_insert", "ext_file_broker"):
    lab.execute("DROP TABLE IF EXISTS " + table)
    lab.execute(history_ddl(table))
lab.execute(f"INSERT INTO ext_file_insert ({','.join(HISTORY_COLUMNS)}) SELECT {','.join(HISTORY_COLUMNS)} FROM {tvf}")
label = "ext_broker_" + uuid4().hex
statement = f'''LOAD LABEL {lab.database}.{label} (
DATA INFILE("{uri}") INTO TABLE ext_file_broker FORMAT AS "parquet"
({','.join(HISTORY_COLUMNS)})
) WITH S3 (
"AWS_ENDPOINT"="http://course-lake-silo:9000","AWS_REGION"="us-east-1",
"AWS_ACCESS_KEY"="{ACCESS_KEY}","AWS_SECRET_KEY"="{SECRET_KEY}","use_path_style"="true"
) PROPERTIES ("timeout"="120","max_filter_ratio"="0")'''
show_sql("Submit batch job", statement)
from dw_course.ui import WorkflowProgress
progress = WorkflowProgress("Broker Load: " + label, {
    1: "Submit this unique load label once",
    2: "Wait for FINISHED (up to 150 seconds; the job timeout is 120 seconds)",
    3: "Compare both loaded copies with every expected field",
})
try:
    progress.advance(1)
    lab.execute(statement)
    progress.advance(2)
    started = time.monotonic()
    deadline = started+150
    while True:
        rows = lab.query("SHOW LOAD WHERE Label = %s", (label,))
        state = rows[0][2] if rows else "Waiting for the label to appear"
        load_progress = rows[0][3] if rows and len(rows[0]) > 3 else "not reported yet"
        now = time.monotonic()
        progress.update(f"Label={label}; State={state}; Progress={load_progress}; "
                        f"elapsed={now-started:.1f}s; remaining={max(0,deadline-now):.1f}s")
        if state == "FINISHED":
            break
        if state == "CANCELLED":
            raise RuntimeError(f"Broker Load cancelled: {rows}")
        if now >= deadline:
            lab.execute("CANCEL LOAD WHERE LABEL = %s", (label,))
            raise TimeoutError(f"Broker Load timeout: {rows}")
        time.sleep(1)
    lab.sql("SHOW LOAD WHERE Label = %s", (label,))
    progress.advance(3)
    for table in ("ext_file_insert", "ext_file_broker"):
        expect(lab.query(f"SELECT {projection} FROM {table} ORDER BY order_id"), history_rows())
except Exception as error:
    progress.fail(str(error))
    raise
progress.finish()

<a id="unit-u.40efed32489141b390ff4215b9ab7c0b"></a>

<a id="3-defaults-and-generated-columns-are-different"></a>

## 3. Әдепкі мәндер мен есептелетін бағандар әртүрлі

status көрсетілмесе, CREATED беріледі; PAID айқын көрсетілсе, кіріс сақталады.
amount мәні amount_cents арқылы есептеледі: 18000 цент 180.00 болады. INSERT есептелетін бағанды бермейді.
Бұл екі жазба — тәуелсіз оқу үлгілері, олар тарихқа не ағымдағы күй кестелеріне қосылмайды.


In [ ]:
lab.execute("DROP TABLE IF EXISTS ext_defaults")
lab.execute('''CREATE TABLE ext_defaults (
order_id BIGINT, amount_cents BIGINT NOT NULL,
status VARCHAR(20) NOT NULL DEFAULT "CREATED",
amount DECIMAL(12,2) AS (CAST(amount_cents AS DECIMAL(12,2)) / 100)
) DUPLICATE KEY(order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1 PROPERTIES("replication_num"="1")''')
lab.execute("INSERT INTO ext_defaults (order_id,amount_cents) VALUES (901001,18000)")
lab.execute("INSERT INTO ext_defaults (order_id,amount_cents,status) VALUES (901002,8000,'PAID')")
expect(lab.query("SELECT order_id,status,amount FROM ext_defaults ORDER BY order_id"), [(901001,"CREATED","180.00"),(901002,"PAID","80.00")])
lab.sql("SELECT * FROM ext_defaults ORDER BY order_id");

<a id="unit-u.262350a932754f36b9137998605314bc"></a>

<a id="4-group-commit-response-time-is-not-query-visibility-time"></a>

## 4. Group Commit жауап уақыты деректердің сұрауға көріну уақыты емес

Бірдей он модельденген тапсырысты үш бос кестеге бөлек жіберіп, жауап келген уақытты және барлық деректер алғаш сұрауға қолжетімді болған уақытты тіркеңіз.
Group Commit сұраулары жеке label бермейді; off_mode label арқылы қайта әрекет жасау қорытындыларын қолданудың орнына қайтарылған GroupCommit өрісі арқылы қолданылған жолды растаңыз.
Алғашқы async сұрауда деректер көріне ме, соны бақылаңыз; «міндетті түрде көрінбеуін» талап етпеңіз және бекітілген кідіріс арақатынасын болжамаңыз.
Бұл жұмыс тек растау мен көрінуді салыстырады; бірнеше сұраудың бір транзакцияны бөлісуі, тұрақты өткізу қабілеті және нұсқалар кезегі үшін бөлек параллель жүктеме тәжірибесі қажет.


In [ ]:
import os
import requests
from dw_course.schema import ORDER_COLUMNS, order_ddl, order_rows
payload = (COURSE_ROOT / "datasets/orders.csv").read_bytes()
endpoint = os.environ["DW_BE_HTTP_URL"].rstrip("/")
measurements = []
for mode in ("off_mode", "sync_mode", "async_mode"):
    table = "ext_gc_" + mode
    lab.execute("DROP TABLE IF EXISTS " + table)
    lab.execute(order_ddl(table))
    lab.execute(f'ALTER TABLE {table} SET ("group_commit_interval_ms"="2000")')
    headers = {"format":"csv", "column_separator":",", "columns":",".join(ORDER_COLUMNS),
               "strict_mode":"true", "max_filter_ratio":"0", "group_commit":mode}
    if mode == "off_mode":
        headers["label"] = "ext_off_" + uuid4().hex
    started = time.monotonic()
    response = requests.put(f"{endpoint}/api/{lab.database}/{table}/_stream_load",
                            auth=(lab.user,lab.password),headers=headers,data=payload,
                            allow_redirects=False,timeout=120)
    response.raise_for_status()
    result = response.json()
    show_response(result, title=mode + " Stream Load response")
    expect(result["Status"], "Success")
    expect(result["NumberLoadedRows"], 10)
    if mode != "off_mode":
        expect(result["GroupCommit"], True)
    acknowledged = time.monotonic()
    first_count = lab.query(f"SELECT COUNT(*) FROM {table}")[0][0]
    count = first_count
    while count != 10:
        if count > 10:
            raise RuntimeError(f"Unexpected duplicate rows: {count}")
        if time.monotonic()-acknowledged > 60:
            raise TimeoutError(f"Data not visible: {result}")
        time.sleep(0.1)
        count = lab.query(f"SELECT COUNT(*) FROM {table}")[0][0]
    visible = time.monotonic()
    projection = ",".join("DATE_FORMAT(event_time,'%Y-%m-%d %H:%i:%s')" if col=="event_time" else col for col in ORDER_COLUMNS)
    expect(lab.query(f"SELECT {projection} FROM {table} ORDER BY order_id"), order_rows(fixture("orders.json")))
    measurements.append({"Mode": mode, "Response ms": round((acknowledged-started)*1000,2),
                         "First full visibility ms": round((visible-started)*1000,2),
                         "Rows in first query after response": first_count})
    lab.sql(f"SHOW TABLETS FROM {table}")
show_records("Group Commit comparison", measurements)

<a id="unit-u.cd1bf60602334550924ecb7e4206efcb"></a>

<a id="5-independent-explanation-and-troubleshooting"></a>

## 5. Өз бетінше түсіндіру және ақауларды анықтау

Файлды тікелей сұрауға CREATE CATALOG қажет емес, ал Iceberg үшін қажет болуының себебін және LOAD LABEL қайтуы неге бірден аяқталуды білдірмейтінін түсіндіріңіз.
Содан кейін әдепкі мәндер мен есептелетін бағандар қашан қолданылатынын және async сәтті жауабының өзі деректерді сұрауға болатынын неге дәлелдемейтінін түсіндіріңіз.
Сәтсіздік кезінде жауапты, label, TxnId және SHOW LOAD сақтаңыз; Group Commit күту уақыты аяқталса, ойланбай қайта жібермес бұрын WAL/тапсырмаларды зерттеңіз.

Дереккөздер: [S3 TVF](https://doris.apache.org/docs/4.x/sql-manual/sql-functions/table-valued-functions/s3/),
[Broker Load](https://doris.apache.org/docs/4.x/data-operate/import/import-way/broker-load-manual/),
[Group Commit](https://doris.apache.org/docs/4.x/data-operate/import/load-best-practices/group-commit-manual/),
[CREATE TABLE](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/table-and-view/table/CREATE-TABLE/).


In [ ]:
lab.close()